# Day 2 - Data Cleaning
**Bluestock Fintech Capstone**
Clean all 10 datasets — parse dates, forward-fill NAV, validate types, save to processed/.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path().resolve().parent
RAW  = ROOT / 'data' / 'raw'
PROC = ROOT / 'data' / 'processed'
PROC.mkdir(parents=True, exist_ok=True)


In [ ]:
# clean nav_history - most important dataset
nav = pd.read_csv(RAW / '02_nav_history.csv')
nav['date'] = pd.to_datetime(nav['date'])
before = len(nav)
nav = nav.drop_duplicates(subset=['amfi_code','date'])
nav = nav.sort_values(['amfi_code','date']).reset_index(drop=True)
print(f'Duplicates removed: {before - len(nav)}')

# forward-fill for weekends/holidays
full_range = pd.date_range(nav['date'].min(), nav['date'].max(), freq='D')
parts = []
for code, group in nav.groupby('amfi_code'):
    g = group.set_index('date').reindex(full_range)
    g['amfi_code'] = code
    g['nav'] = g['nav'].ffill()
    g = g.dropna(subset=['nav'])
    g.index.name = 'date'
    parts.append(g.reset_index())
nav_clean = pd.concat(parts, ignore_index=True)
nav_clean['nav_pct_change'] = nav_clean.groupby('amfi_code')['nav'].pct_change() * 100
nav_clean['nav_anomaly']    = nav_clean['nav_pct_change'].abs() > 20
print(f'After ffill: {len(nav_clean):,} rows | NAV<=0: {(nav_clean["nav"]<=0).sum()}')
nav_clean.to_csv(PROC / '02_nav_history_clean.csv', index=False)
print('Saved: 02_nav_history_clean.csv')


In [ ]:
# clean investor transactions
tx = pd.read_csv(RAW / '08_investor_transactions.csv')
tx['transaction_date'] = pd.to_datetime(tx['transaction_date'], errors='coerce')
tx['transaction_type'] = tx['transaction_type'].str.strip().str.replace('Sip','SIP')
bad = (tx['amount_inr'] <= 0).sum()
print(f'Bad amounts: {bad}')
tx = tx[tx['amount_inr'] > 0].drop_duplicates().sort_values('transaction_date').reset_index(drop=True)
tx.to_csv(PROC / '08_investor_transactions_clean.csv', index=False)
print(f'Saved: 08_investor_transactions_clean.csv | {len(tx):,} rows')
print(f'transaction_type values: {tx["transaction_type"].unique()}')
print(f'kyc_status values: {tx["kyc_status"].unique()}')


In [ ]:
# clean remaining 8 datasets
import os

mappings = [
    ('01_fund_master.csv',          '01_fund_master_clean.csv',          'launch_date',       None),
    ('03_aum_by_fund_house.csv',    '03_aum_by_fund_house_clean.csv',    'date',              None),
    ('04_monthly_sip_inflows.csv',  '04_monthly_sip_inflows_clean.csv',  'month',             '-01'),
    ('05_category_inflows.csv',     '05_category_inflows_clean.csv',     'month',             '-01'),
    ('06_industry_folio_count.csv', '06_industry_folio_count_clean.csv', 'month',             '-01'),
    ('07_scheme_performance.csv',   '07_scheme_performance_clean.csv',   None,                None),
    ('09_portfolio_holdings.csv',   '09_portfolio_holdings_clean.csv',   'portfolio_date',    None),
    ('10_benchmark_indices.csv',    '10_benchmark_indices_clean.csv',    'date',              None),
]
for src, dst, datecol, suffix in mappings:
    df = pd.read_csv(RAW / src)
    df = df.drop_duplicates()
    str_cols = df.select_dtypes(include='object').columns
    for c in str_cols: df[c] = df[c].str.strip()
    if datecol:
        col_val = df[datecol].astype(str) + (suffix or '')
        df[datecol] = pd.to_datetime(col_val, errors='coerce')
    df.to_csv(PROC / dst, index=False)
    print(f'Saved: {dst} | {len(df)} rows')


In [ ]:
# verify all 10 clean files exist
clean_files = list(PROC.glob('*_clean.csv'))
print(f'Clean files in processed/: {len(clean_files)}')
for f in sorted(clean_files):
    df = pd.read_csv(f)
    print(f'  {f.name}: {df.shape}')
